# Kaggle evaluation run — finalproject_cpe027

Runs the Tier 1 (ACL6060 + seeded DEMAND noise) and Tier 2 (unmodified M3AV) ASR evaluation
on Kaggle (GPU) using preprocessed local data. No preprocessing is redone here.

- Tiers are executed and reported separately; they are never pooled.
- Checkpoints are retained per tier; this notebook starts fresh (`--no-resume`).
- Whisper: `base`, `cuda`, `float16`, `en` (Tier 1). DSP/decoding settings come from the runner.
- Data mounts are discovered recursively (nested under /kaggle/input/datasets/<user>/...).


In [91]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

# ---- resolve project code (attached dataset first, git fallback) ---------
REPO_DIR = None
for p in Path("/kaggle/input").rglob("backend"):
    if (p / "app").is_dir():
        REPO_DIR = p.parent
        break
REPO_DIR = REPO_DIR or Path("/kaggle/working/finalproject_cpe027")
BACKEND_DIR = REPO_DIR / "backend"

REPO_GIT_URL = os.environ.get(
    "REPO_GIT_URL", "https://github.com/leonpscl/finalproject_cpe027.git"
).strip()
if not REPO_DIR.is_dir():
    print("Cloning project repository ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_GIT_URL, str(REPO_DIR)], check=True
    )
if not (BACKEND_DIR / "app").is_dir():
    raise FileNotFoundError(
        "Backend directory not found. Attach a dataset containing backend/app "
        "or make the repository public and ensure git can clone it."
    )

# ---- resolve data mounts recursively (nested under datasets/<user>) ----
INPUT = Path("/kaggle/input")
SEARCH_ROOTS = [INPUT, *[p for p in INPUT.glob("datasets/*") if p.is_dir()],
                 *[p for p in INPUT.glob("datasets/*/*") if p.is_dir()]]

acl_hits = []
for p in SEARCH_ROOTS:
    cand = (p / "dev" / "full_wavs")
    if cand.is_dir():
        acl_hits.append(p)
acl_hits = list(dict.fromkeys(acl_hits))
if not acl_hits:
    listing = sorted(str(p.name) for p in INPUT.iterdir()) if INPUT.is_dir() else []
    raise FileNotFoundError(
        "ACL6060 data not found. Scanned roots: " + ", ".join(str(p) for p in SEARCH_ROOTS)
        + ". Top-level entries: " + ", ".join(listing)
    )

demand_hits = []
for p in SEARCH_ROOTS:
    for cand in (p, p / "DEMAND"):
        if (cand / "DKITCHEN_16k").is_dir() and next(cand.rglob("*.wav"), None) is not None:
            demand_hits.append(cand)
demand_hits = list(dict.fromkeys(demand_hits))
if not demand_hits:
    listing = sorted(str(p.name) for p in INPUT.iterdir()) if INPUT.is_dir() else []
    raise FileNotFoundError(
        "DEMAND data not found. Scanned roots: " + ", ".join(str(p) for p in SEARCH_ROOTS)
        + ". Top-level entries: " + ", ".join(listing)
    )

m3av_raw = sorted({str(p / "M3AV" / "extracted")
                for p in SEARCH_ROOTS
                if (p / "M3AV" / "extracted").is_dir()})
m3av_zips = sorted({z for p in SEARCH_ROOTS for z in p.rglob("*.zip") if "m3av" in z.name.lower()})

ACL_ROOT = acl_hits[0]
DEMAND_ROOT = demand_hits[0]

os.environ["REPO_DIR"] = str(REPO_DIR)
os.environ["BACKEND_DIR"] = str(BACKEND_DIR)
os.environ["ACL_ROOT"] = str(ACL_ROOT)
os.environ["DEMAND_ROOT"] = str(DEMAND_ROOT)
M3AV_ZIPS = [str(z) for z in m3av_zips]
os.environ["M3AV_ZIPS"] = chr(10).join(M3AV_ZIPS)
os.environ["M3AV_CLIP_ROOT"] = "\n".join(m3av_raw)

print("ACL root:", ACL_ROOT)
print("DEMAND root:", DEMAND_ROOT)
print("M3AV raw roots:", m3av_raw)
print("M3AV zips (lazy-extract):", [z.name for z in m3av_zips], flush=True)
!nvidia-smi


ACL root: /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-acl6060
DEMAND root: /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-demand/DEMAND
M3AV raw roots: []
M3AV zips (lazy-extract): []
Thu Nov  5 22:38:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8

In [92]:
!df -h

Filesystem                        Size  Used Avail Use% Mounted on
overlay                           8.0T  7.1T  967G  89% /
tmpfs                              64M     0   64M   0% /dev
shm                                14G     0   14G   0% /dev/shm
/dev/sda1                         122G  6.0G  116G   5% /opt/bin
/dev/loop2                         20G  397M   20G   2% /kaggle/lib
kkb-production-mirror-asia-east1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-acl6060
kkb-production-mirror-asia-east1  1.0P     0  1.0P   0% /kaggle/input/datasets/leonpascual1/finalproject-cpe027-eval-demand
/dev/mapper/snap                  8.0T  7.1T  967G  89% /etc/hosts
tmpfs                             4.0K     0  4.0K   0% /proc/acpi


In [93]:
from pathlib import Path

print("Input directory tree (up to depth 3):")
for p in sorted(Path("/kaggle/input").rglob("*")):
    if not p.is_dir():
        continue
    rel = p.relative_to("/kaggle/input")
    if len(rel.parts) <= 3:
        print("  " * (len(rel.parts) - 1) + rel.name + "/")


Input directory tree (up to depth 3):
datasets/
  leonpascual1/
    finalproject-cpe027-eval-acl6060/
    finalproject-cpe027-eval-demand/


In [94]:
!pip install -q faster-whisper av scipy numpy rapidfuzz matplotlib pydantic-settings python-dotenv httpx pydantic

In [95]:
os.environ["ENV"] = "production"
os.environ["PROCESSING_MODE"] = "real" 
import os

os.environ["WHISPER_MODEL"] = "base"
os.environ["WHISPER_DEVICE"] = "cuda"
os.environ["WHISPER_COMPUTE_TYPE"] = "float16"
os.environ["WHISPER_THREADS"] = "4"
os.environ["WHISPER_LANGUAGE"] = "en"
os.environ["MODEL_CACHE_DIR"] = "/kaggle/working/models"
os.environ["EVAL_DATA_ROOT"] = os.environ.get("ACL_ROOT", "/kaggle/input/acl6060")


In [98]:
import subprocess
import sys
from importlib.metadata import version


def av_ok() -> bool:
    """Probe in a fresh interpreter so we see the currently installed av."""
    probe = subprocess.run(
        [sys.executable, "-c",
         "import av; av.open('/nonexistent.wav', metadata_errors='ignore')"],
        capture_output=True, text=True,
    )
    last = (probe.stderr.strip().splitlines() or [""])[-1]
    return "unexpected keyword argument 'metadata_errors'" not in last


if not av_ok():
    print(f"av {version('av')} is incompatible with faster-whisper "
          f"{version('faster-whisper')}; trying older prebuilt wheels ...")
    fixed = False
    # Newest prebuilt wheel below each ceiling; never compile from source.
    for ceiling in ["19", "17", "16", "15", "14", "13", "12"]:
        spec = f"av<{ceiling}"
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "--only-binary=:all:", spec],
            capture_output=True, text=True,
        )
        if r.returncode != 0:
            print(f"  {spec}: no usable wheel")
            continue
        ok = av_ok()
        print(f"  {spec}: installed av {version('av')} -> {'OK' if ok else 'still incompatible'}")
        if ok:
            fixed = True
            break
    if not fixed:
        raise RuntimeError(
            "No prebuilt av wheel works with this faster-whisper. Options: "
            "(1) `!python -m pip install -q 'faster-whisper<1.1'` and re-run this cell, or "
            "(2) paste this output so we can pick another pin."
        )
print("av/faster-whisper compatibility: OK\n")

for pkg in ["ctranslate2", "faster-whisper", "av", "scipy", "numpy", "pandas",
            "rapidfuzz", "matplotlib", "pydantic", "pydantic_settings"]:
    try:
        print(f"{pkg}: {version(pkg)}")
    except Exception:
        print(f"{pkg}: unknown")

try:
    import torch
    print("torch:", torch.__version__)
except Exception:
    print("torch: not installed")

av 19.0.2 is incompatible with faster-whisper 1.2.1; trying older prebuilt wheels ...
  av<19: installed av 18.1.0 -> OK
av/faster-whisper compatibility: OK

ctranslate2: 4.8.5
faster-whisper: 1.2.1
av: 18.1.0
scipy: 1.16.3
numpy: 2.1.3
pandas: 2.3.3
rapidfuzz: 3.14.6
matplotlib: 3.10.0
pydantic: 2.14.1
pydantic_settings: 2.16.2
torch: 2.11.0+cu130


## Tier 1 smoke test


In [101]:
!pip install -q faster-whisper "av==18.1.0" scipy numpy rapidfuzz matplotlib pydantic-settings python-dotenv httpx pydantic
from importlib.metadata import version
print("av:", version("av"), "| faster-whisper:", version("faster-whisper"))

av: 18.1.0 | faster-whisper: 1.2.1


In [104]:
import glob
import os
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "nvidia-cublas-cu12", "nvidia-cudnn-cu12"], check=True)

# Locate the installed libs and prepend them to LD_LIBRARY_PATH.
# Shell commands (!python ...) inherit this, so the runner subprocess sees it.
site = next(p for p in sys.path if p.endswith("dist-packages") and os.path.isdir(os.path.join(p, "nvidia")))
lib_dirs = glob.glob(os.path.join(site, "nvidia", "cublas", "lib")) + \
           glob.glob(os.path.join(site, "nvidia", "cudnn", "lib"))
print("lib dirs:", lib_dirs)

os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])

print(glob.glob(os.path.join(site, "nvidia", "cublas", "lib", "libcublas.so.12*")))

lib dirs: ['/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib', '/usr/local/lib/python3.13/dist-packages/nvidia/cudnn/lib']
['/usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12']


In [105]:
import os
from pathlib import Path

ACL_ROOT = os.environ.get("ACL_ROOT")
DEMAND_ROOT = os.environ.get("DEMAND_ROOT")
BACKEND_DIR = os.environ.get("BACKEND_DIR", "/kaggle/working/finalproject_cpe027/backend")

if not (ACL_ROOT and DEMAND_ROOT and Path(ACL_ROOT).is_dir() and Path(DEMAND_ROOT).is_dir()):
    raise RuntimeError("Attach ACL6060 and DEMAND datasets before the smoke test.")

os.chdir(BACKEND_DIR)
!python -m app.eval_runner --tier 1 --limit 1 --split eval --conditions arm_a,hp --snrs 0 --noise demand --demand-envs DKITCHEN,DLIVING,DWASHING --data-root "$ACL_ROOT" --demand-root "$DEMAND_ROOT" --out /kaggle/working/eval_smoke_tier1


Loading ASR model...
Tier 1: 5 talks, SNRs=[0.0]
  2022.acl-long.111 | demand:DKITCHEN | SNR=0 dB | arm_a
    ASR segments: 1 (0.6-4.6s)
    ASR segments: 10 (43.6-46.6s)
    ASR segments: 20 (97.2-106.3s)
    ASR segments: 30 (176.5-180.7s)
    ASR segments: 40 (242.8-249.8s)
    ASR segments: 50 (325.3-331.3s)
    ASR segments: 60 (384.0-399.2s)
    ASR segments: 70 (490.9-497.3s)
    ASR segments: 80 (558.0-564.9s)
    ASR segments: 90 (610.8-623.2s)
    ASR segments: 100 (697.1-705.6s)
    WER 26.7% | CER 10.9% | TermPresence 81.2% | RTF 0.05 | 38.8s
  2022.acl-long.111 | demand:DKITCHEN | SNR=0 dB | hp
    ASR segments: 1 (0.6-4.6s)
    ASR segments: 10 (53.7-61.0s)
    ASR segments: 20 (104.6-108.0s)
    ASR segments: 30 (139.1-145.5s)
    ASR segments: 40 (198.1-202.1s)
    ASR segments: 50 (267.4-275.6s)
    ASR segments: 60 (342.3-344.9s)
    ASR segments: 70 (377.7-381.5s)
    ASR segments: 80 (429.0-434.3s)
    ASR segments: 90 (499.4-510.8s)
    ASR segments: 100 (583.8-587

In [107]:
import json
from pathlib import Path

t1 = json.loads(Path("/kaggle/working/eval_smoke_tier1/tier1_acl_results.json").read_text())
print("SMOKE tier1 records:", len(t1))
r = t1[0]
print(f"SMOKE WER: {r["wer"]:.2%} | RTF: {r["rtf"]:.3f} | seconds: {r["seconds"]:.1f}")


SMOKE tier1 records: 6
SMOKE WER: 26.69% | RTF: 0.054 | seconds: 38.8


## Full run  Tier 1 (ACL6060 + seeded DEMAND noise)

All six conditions, SNRs `none,0,5,10,20`, DEMAND envs `DKITCHEN,DLIVING,DWASHING`. Fresh run (`--no-resume`).

In [108]:
!grep -n -i -E "resume|checkpoint" /kaggle/working/finalproject_cpe027/backend/app/eval_runner.py | head -40

113:CHECKPOINT_SCHEMA_VERSION = 2
118:class CheckpointIncompatibleError(RuntimeError):
119:    """Raised when a checkpoint cannot be safely resumed for the current run."""
200:def _load_checkpoint(path: Path, expected_manifest: dict[str, Any] | None = None) -> dict[str, Any]:
202:        return {"schema_version": CHECKPOINT_SCHEMA_VERSION, "records": [], "run_manifest": expected_manifest}
207:        raise CheckpointIncompatibleError(f"incompatible checkpoint (unreadable): {exc}") from exc
209:        raise CheckpointIncompatibleError("incompatible checkpoint (missing records list)")
211:    if schema_version != CHECKPOINT_SCHEMA_VERSION:
212:        raise CheckpointIncompatibleError(
213:            f"incompatible checkpoint schema {schema_version}; "
214:            "start a new run with --no-resume or remove the checkpoint"
217:        raise CheckpointIncompatibleError("incompatible checkpoint manifest/fingerprint")
219:        # Existing checkpoint must be validated by its own mani

In [109]:
import os
from pathlib import Path

ACL_ROOT = os.environ.get("ACL_ROOT")
DEMAND_ROOT = os.environ.get("DEMAND_ROOT")
if not ACL_ROOT or not Path(ACL_ROOT).is_dir():
    raise RuntimeError("Attach ACL6060 data before Tier 1")
if not DEMAND_ROOT or not Path(DEMAND_ROOT).is_dir():
    raise RuntimeError("Attach DEMAND data before Tier 1")
os.chdir(os.environ.get("BACKEND_DIR", "/kaggle/working/finalproject_cpe027/backend"))
!python -m app.eval_runner --tier 1 --split eval --conditions arm_a,hp,hp+sg,hp_ffmpeg,hp_wiener,hp_wiener_oa --snrs none,0,5,10,20 --noise demand --demand-envs DKITCHEN,DLIVING,DWASHING --data-root "$ACL_ROOT" --demand-root "$DEMAND_ROOT" --out /kaggle/working/batches/tier1_eval


Loading ASR model...
Tier 1: 5 talks, SNRs=[None, 0.0, 5.0, 10.0, 20.0]
  2022.acl-long.111 | demand:DKITCHEN | SNR=clean | arm_a
    ASR segments: 1 (0.6-5.1s)
    ASR segments: 10 (28.0-31.2s)
    ASR segments: 20 (66.8-71.0s)
    ASR segments: 30 (118.3-121.2s)
    ASR segments: 40 (155.9-159.6s)
    ASR segments: 50 (188.7-191.1s)
    ASR segments: 60 (236.6-243.2s)
    ASR segments: 70 (297.3-302.3s)
    ASR segments: 80 (367.9-372.6s)
    ASR segments: 90 (422.8-428.8s)
    ASR segments: 100 (503.6-512.8s)
    ASR segments: 110 (578.8-583.8s)
    ASR segments: 120 (641.1-651.6s)
    WER 26.1% | CER 9.4% | TermPresence 79.7% | RTF 0.05 | 38.3s
  2022.acl-long.111 | demand:DKITCHEN | SNR=clean | hp
    ASR segments: 1 (0.6-5.1s)
    ASR segments: 10 (27.9-30.5s)
    ASR segments: 20 (75.5-81.8s)
    ASR segments: 30 (147.3-156.0s)
    ASR segments: 40 (215.4-224.7s)
    ASR segments: 50 (267.1-271.2s)
    ASR segments: 60 (307.8-309.7s)
    ASR segments: 70 (354.9-357.3s)
    ASR s

## Tier 1 analysis  WER/CER vs DSP condition (paired by talk and noise/SNR)

All analyses below are Tier-1-only. Each figure is written under `/kaggle/working/figures/tier1/` so the
notebook root stays clean. Raw results are written first in `/kaggle/working/batches/tier1_eval`. Figures and
tabular summaries are saved after reading the batch CSV so the notebook remains resumable.


In [110]:

from pathlib import Path
from pathlib import Path as _Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

FIGURES = Path("/kaggle/working/figures/tier1")
FIGURES.mkdir(parents=True, exist_ok=True)
RESULT = Path("/kaggle/working/batches/tier1_eval/tier1_acl_results.csv").resolve()

if not RESULT.exists():
    print(f"Tier 1 results not found: {RESULT}")
else:
    t1 = pd.read_csv(RESULT)
    print("TIER 1 rows:", len(t1), "\n")

    summary = (
        t1.groupby("condition", dropna=False)[["wer", "cer", "rtf", "seconds", "signal_snr_improvement_db", "term_presence_recall"]]
           .agg(["mean", "min", "max", "median"])
           .round(5)
    )
    print("OVERALL per condition:")
    print(summary)

    print("\nRELATIVE ASR GAIN vs arm_a, grouped by (noise_type, snr_db):")
    for key, sub in t1.groupby(["noise_type", "snr_db"]):
        base = sub.loc[sub.condition.eq("arm_a"), "wer"].mean() if (sub.condition == "arm_a").any() else float("nan")
        if pd.isna(base) or base <= 0:
            continue
        deltas = (sub.groupby("condition")["wer"].mean() - base) / base * 100.0
        deltas["__arm_a_baseline__"] = 0.0
        print(f"\n-- {key} -- arm_a mean WER = {base:.4f}")
        print(deltas.sort_values().round(2).to_string())

    # Figure 1: mean WER per condition.
    means = t1.groupby("condition", sort=True)["wer"].mean()
    errs = t1.groupby("condition", sort=True)["wer"].std(ddof=1).fillna(0.0)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(means.index, means.values, yerr=errs.values, capsize=4, color="#4c78a8")
    ax.set_title("Tier 1  mean WER by DSP condition")
    ax.set_xlabel("DSP condition")
    ax.set_ylabel("WER (lower is better)")
    ax.set_xticks(range(len(means.index.tolist())))
    ax.set_xticklabels(means.index.tolist(), rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier1_wer_by_condition.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Figure 2: relative WER gain vs arm_a under seeded DEMAND noise.
    baselines = t1.loc[t1.condition == "arm_a", ["noise_type", "snr_db", "talk_id", "wer"]]
    baselines = baselines.rename(columns={"wer": "arm_a_wer"})
    delta_rows = t1.merge(baselines, on=["noise_type", "snr_db", "talk_id"], how="inner")
    delta_rows["delta_vs_arm_a"] = (delta_rows["wer"] - delta_rows["arm_a_wer"]) / delta_rows["arm_a_wer"].clip(lower=1e-9) * 100.0
    delta_mean = delta_rows.groupby("condition")["delta_vs_arm_a"].mean().sort_values()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(delta_mean.index, delta_mean.values, color="#d6604d")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("Tier 1  mean WER change vs arm_a")
    ax.set_xlabel("DSP condition")
    ax.set_ylabel("Relative WER change (%)")
    ax.set_xticks(range(len(delta_mean.index.tolist())))
    ax.set_xticklabels(delta_mean.index.tolist(), rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier1_wer_gain_vs_arm_a.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Figure 3: SNR improvement vs WER change.
    noisy = delta_rows.loc[delta_rows.noise_type != "none", :]
    if "signal_snr_improvement_db" in noisy.columns and noisy["signal_snr_improvement_db"].notna().any():
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.scatter(noisy["signal_snr_improvement_db"], noisy["delta_vs_arm_a"], alpha=0.45)
        ax.set_title("Tier 1  signal SNR improvement vs WER change")
        ax.set_xlabel("Signal SNR improvement (dB)")
        ax.set_ylabel("WER change vs arm_a (%)")
        fig.tight_layout()
        fig.savefig(FIGURES / "tier1_snr_improvement_vs_wer.png", dpi=150, bbox_inches="tight")
        plt.close(fig)
        print("Saved Tier 1 figures to", FIGURES)
    else:
        print("No signal SNR improvement values found; skipped Tier 1 SNR figure")

    # Figure 4: RTF vs WER tradeoff.
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(delta_rows.groupby("condition")[["wer", "rtf"]].mean()["rtf"].values,
               delta_rows.groupby("condition")[["wer", "rtf"]].mean()["wer"].values)
    for cond, point in delta_rows.groupby("condition")[["wer", "rtf"]].mean().iterrows():
        ax.annotate(cond, (point["rtf"], point["wer"]))
    ax.set_title("Tier 1  RTF vs WER tradeoff")
    ax.set_xlabel("RTF (higher = more compute)")
    ax.set_ylabel("WER")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier1_rtf_vs_wer.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved Tier 1 figures to", FIGURES)


Tier 1 results not found: /kaggle/working/batches/tier1_eval/tier1_acl_results.csv


## Tier 2 smoke test


In [ ]:
import os, shutil, zipfile
from pathlib import Path

BACKEND_DIR = os.environ.get("BACKEND_DIR", "/kaggle/working/finalproject_cpe027/backend")
M3AV_ROOT = os.environ.get("M3AV_CLIP_ROOT", "").splitlines()[0] if os.environ.get("M3AV_CLIP_ROOT", "").splitlines() else ""

if not (M3AV_ROOT and Path(M3AV_ROOT).is_dir()):
    zips = [z for z in os.environ.get("M3AV_ZIPS", "").splitlines() if z.strip()]
    if not zips:
        raise RuntimeError("Attach at least one M3AV split dataset before the smoke test.")
    stamp = Path("/kaggle/working/mounts/_smoke_m3av")
    if not (stamp / "M3AV" / "extracted").is_dir():
        if stamp.exists(): shutil.rmtree(stamp)
        stamp.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z: z.extractall(stamp)
    M3AV_ROOT = str(stamp / "M3AV" / "extracted")

os.chdir(BACKEND_DIR)
!python -m app.eval_runner --tier 2 --conditions arm_a,hp --snrs 0 --m3av-root "$M3AV_ROOT" --out /kaggle/working/eval_smoke_tier2


In [ ]:
import json
from pathlib import Path

t2 = json.loads(Path("/kaggle/working/eval_smoke_tier2/tier2_m3av_results.json").read_text())
print("SMOKE tier2 records:", len(t2))
r2 = t2[0]
print(f"SMOKE tier2 WER: {r2["wer"]:.2%} | RTF: {r2["rtf"]:.3f} | NSHR: {r2["nshr"]}")


## Full run  Tier 2 (M3AV), batched per duration bucket

In [ ]:
import os
import shutil
import sys
import zipfile
from pathlib import Path
from types import SimpleNamespace

BACKEND_DIR = os.environ.get("BACKEND_DIR", os.path.join("/kaggle/working", "finalproject_cpe027", "backend"))
os.chdir(BACKEND_DIR)
sys.path.insert(0, BACKEND_DIR)
from app import eval_data, eval_runner

INPUT = Path("/kaggle/input")
raw_roots = sorted({
    str(p.parent.parent) for p in INPUT.rglob("extracted")
    if p.is_dir() and p.parent.name == "M3AV"
})
env_roots = [r for r in os.environ.get("M3AV_CLIP_ROOT", "").split(chr(10)) if r.strip()]
m3av_zips = [z for z in os.environ.get("M3AV_ZIPS", "").split(chr(10)) if z.strip()]

if raw_roots:
    inputs = raw_roots
elif env_roots:
    inputs = env_roots
elif m3av_zips:
    inputs = m3av_zips
else:
    raise RuntimeError("No M3AV splits are attached; Tier 2 cannot run.")

CONDITIONS = "arm_a,hp,hp+sg,hp_ffmpeg,hp_wiener,hp_wiener_oa"
BUCKETS = ["<5m", "5-10m", "10-20m", ">=20m"]
MOUNTS = Path("/kaggle/working/mounts")

def extracted_root(zip_path):
    dest = MOUNTS / ("m3av_" + Path(zip_path).stem)
    marker = dest / "M3AV" / "extracted"
    if not marker.is_dir():
        if dest.exists():
            shutil.rmtree(dest)
        dest.mkdir(parents=True, exist_ok=True)
        print("Extracting", Path(zip_path).name, flush=True)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dest)
    return dest / "M3AV" / "extracted"

for src in inputs:
    if str(src).endswith(".zip"):
        clip_root = extracted_root(src)
        zip_source = src
    else:
        clip_root = Path(src)
        zip_source = None
        if not clip_root.is_dir():
            print("Skip missing root:", clip_root)
            continue

    for bucket in BUCKETS:
        selected = [c for c in eval_data.load_m3av_clips(str(clip_root))
                    if eval_data.duration_bucket(c.duration_seconds) == bucket]
        if not selected:
            continue
        out_name = Path(src).stem if str(src).endswith(".zip") else clip_root.name
        out_dir = "/kaggle/working/batches/tier2_" + \
            bucket.replace("<", "lt_").replace(">", "gt_").replace("+", "p") + "_" + out_name
        orig_loader = eval_runner.load_m3av_clips
        eval_runner.load_m3av_clips = lambda data_root=None, _sel=selected: _sel
        args = SimpleNamespace(
            tier="2",
            conditions=CONDITIONS,
            m3av_root=str(clip_root),
            data_root=None,
            out=out_dir,
            limit=None,
            resume=True,
        )
        try:
            records, _summary = eval_runner.run_tier2(args)
        finally:
            eval_runner.load_m3av_clips = orig_loader
        print(
            "tier2 %s bucket %s: %d records -> %s"
            % (clip_root.name, bucket, len(records), out_dir)
        )

    if zip_source is not None:
        shutil.rmtree(clip_root.parent.parent, ignore_errors=True)
        print("Cleaned", clip_root.parent.parent, flush=True)


## Tier 2 analysis  per duration bucket and overall aggregate

Tier 2 evidence is M3AV-only. All figures are written under `/kaggle/working/figures/tier2/`.
Each bucket is plotted separately from its own batch output; no pooling occurs with Tier 1.


In [ ]:

from pathlib import Path
import glob
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGURES = Path("/kaggle/working/figures/tier2")
FIGURES.mkdir(parents=True, exist_ok=True)

paths = glob.glob("/kaggle/working/batches/tier2_*/tier2_m3av_results.csv")
if not paths:
    print("Tier 2 results not found yet; expected under /kaggle/working/batches/tier2_*/tier2_m3av_results.csv")
else:
    df = pd.concat((pd.read_csv(p) for p in paths), ignore_index=True)
    print("TIER 2 rows:", len(df))
    print("OVERALL per condition:")
    print(df.groupby("condition")[["wer", "cer", "rtf", "seconds", "nshr"]].mean().round(5))

    print("\nPER BUCKET x CONDITION:")
    print(df.groupby(["duration_bucket", "condition"])[["wer", "cer", "rtf", "seconds", "nshr"]].mean().round(5))

    print("\nRELATIVE WER vs arm_a:")
    for bucket, sub in df.groupby("duration_bucket"):
        base = sub.loc[sub.condition == "arm_a", "wer"].mean() if (sub.condition == "arm_a").any() else float("nan")
        if pd.isna(base) or base <= 0:
            continue
        deltas = (sub.groupby("condition")["wer"].mean() - base) / base * 100.0
        deltas["__arm_a_baseline__"] = 0.0
        print(f"\n-- {bucket} -- arm_a mean WER = {base:.4f}")
        print(deltas.sort_values().round(2).to_string())

    # Figure 1: mean WER by condition.
    means = df.groupby("condition", sort=True)["wer"].mean()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(means.index, means.values, color="#4c78a8")
    ax.set_title("Tier 2  mean WER by DSP condition")
    ax.set_xlabel("DSP condition")
    ax.set_ylabel("WER (lower is better)")
    ax.set_xticks(range(len(means.index.tolist())))
    ax.set_xticklabels(means.index.tolist(), rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier2_wer_by_condition.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Figure 2: WER by condition and duration bucket.
    bucket_means = df.groupby(["duration_bucket", "condition"])["wer"].mean().unstack("condition")
    ax = bucket_means.plot(kind="bar", figsize=(11, 6))
    ax.set_title("Tier 2  WER by condition and duration bucket")
    ax.set_xlabel("Duration bucket")
    ax.set_ylabel("WER")
    fig = ax.get_figure()
    fig.tight_layout()
    fig.savefig(FIGURES / "tier2_wer_by_bucket.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Figure 3: WER gain vs arm_a by condition and duration bucket.
    fig, ax = plt.subplots(figsize=(11, 6))
    for bucket, sub in df.groupby("duration_bucket"):
        base = sub.loc[sub.condition == "arm_a", "wer"].mean() if (sub.condition == "arm_a").any() else float("nan")
        if pd.isna(base) or base <= 0:
            continue
        gain = (sub.groupby("condition")["wer"].mean() - base) / base * 100.0
        ax.plot(gain, label=bucket, marker="o")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title("Tier 2  relative WER gain vs arm_a")
    ax.set_xlabel("DSP condition")
    ax.set_ylabel("Relative WER change (%)")
    ax.set_xticks(range(len(gain.index.tolist())))
    ax.set_xticklabels(gain.index.tolist(), rotation=30, ha="right")
    ax.legend(title="Duration bucket")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier2_wer_gain_vs_arm_a.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Figure 4: NSHR by condition.
    nshr = df.groupby("condition")["nshr"].mean()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(nshr.index, nshr.values, color="#4c78a8")
    ax.set_title("Tier 2  mean NSHR by condition")
    ax.set_xlabel("DSP condition")
    ax.set_ylabel("NSHR (non-speech hallucination rate)")
    ax.set_xticks(range(len(nshr.index.tolist())))
    ax.set_xticklabels(nshr.index.tolist(), rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier2_nshr_by_condition.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Figure 5: RTF vs WER tradeoff.
    trade = df.groupby("condition")[["wer", "rtf"]].mean()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(trade["rtf"], trade["wer"])
    for cond, point in trade.iterrows():
        ax.annotate(cond, (point["rtf"], point["wer"]))
    ax.set_title("Tier 2  RTF vs WER tradeoff")
    ax.set_xlabel("RTF (higher = more compute)")
    ax.set_ylabel("WER")
    fig.tight_layout()
    fig.savefig(FIGURES / "tier2_rtf_vs_wer.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    print("Saved Tier 2 figures to", FIGURES)


## Merge batch outputs into final results

In [ ]:
import os
from pathlib import Path

REPO_DIR = os.environ.get("REPO_DIR", "/kaggle/working/finalproject_cpe027")
MERGE_SCRIPT = Path(REPO_DIR) / "kaggle" / "merge_eval_results.py"
if not MERGE_SCRIPT.is_file():
    MERGE_SCRIPT = Path(REPO_DIR) / "merge_eval_results.py"
os.chdir(REPO_DIR)

!python {MERGE_SCRIPT} --batch-dir /kaggle/working/batches --out /kaggle/working/final


In [ ]:
from pathlib import Path

import json

names = [
    "tier1_acl_results.json",
    "tier1_acl_results.csv",
    "tier1_acl_summary.json",
    "tier2_m3av_results.json",
    "tier2_m3av_results.csv",
    "tier2_m3av_summary.json",
]
final = Path("/kaggle/working/final")
for name in names:
    p = final / name
    print(name, "OK" if p.is_file() else f"MISSING: {p}")

t1_path = final / "tier1_acl_results.json"
t2_path = final / "tier2_m3av_results.json"
if not t1_path.is_file() or not t2_path.is_file():
    raise RuntimeError("Final tier JSON files not found; merge step may have failed")
t1 = json.loads(t1_path.read_text(encoding="utf-8"))
t2 = json.loads(t2_path.read_text(encoding="utf-8"))
print("Tier 1 records:", len(t1))
print("Tier 2 records:", len(t2))
if len(t1) == 0:
    raise RuntimeError("Final Tier 1 JSON has 0 records")
if len(t2) == 0:
    raise RuntimeError("Final Tier 2 JSON has 0 records")
